In [ ]:
# ── CELL 1: Cài thư viện ──────────────────────────────────────────────────────
!pip install transformers peft pyarrow accelerate underthesea -q

In [ ]:
# ── CELL 2: Imports ───────────────────────────────────────────────────────────
import os, json, time, math
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from peft import get_peft_model, LoraConfig, TaskType

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

print('✅ Imports OK')
print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ── CELL 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
# ── CELL 4: CONFIG – chỉnh ở đây ─────────────────────────────────────────────
CONFIG = {
    # === Paths ===
    # Đặt đúng path file parquet trên Drive
    "data_path"  : "/content/drive/MyDrive/scam_data/processed.parquet",
    # Thư mục lưu checkpoint + log + best model
    "output_dir" : "/content/drive/MyDrive/scam_data/checkpoints",

    # === Model ===
    "model_name"     : "vinai/phobert-base",
    "max_length"     : 256,      # token limit – tăng lên 512 nếu RAM cho phép
    "lora_r"         : 16,
    "lora_alpha"     : 32,
    "lora_dropout"   : 0.1,

    # === Training ===
    "batch_size"     : 32,
    "learning_rate"  : 2e-5,
    "epochs"         : 10,
    "warmup_ratio"   : 0.05,     # 5% steps đầu warmup
    "grad_clip"      : 1.0,

    # === Focal Loss – quan trọng vì data cực mất cân bằng ===
    # alpha = weight class Scam=True (0.75 → ưu tiên recall scam)
    "focal_gamma"    : 2.0,
    "focal_alpha"    : 0.85,

    # === Checkpointing ===
    # Lưu mỗi N steps (không phải mỗi epoch)
    # Với 34M rows / batch 32 ≈ 1M steps/epoch → nên set 5000
    "save_every_steps": 5000,

    # === Early Stopping ===
    "early_stop_patience": 3,    # dừng nếu F1 không tăng sau 3 lần validate
    "eval_every_steps"   : 5000, # validate mỗi N steps

    # === Data split ===
    "val_size"  : 0.05,
    "test_size" : 0.05,
    "seed"      : 42,
}

# Tạo output dir
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
CKPT_PATH     = os.path.join(CONFIG["output_dir"], "checkpoint.pt")
BEST_MODEL_PATH = os.path.join(CONFIG["output_dir"], "best_model.pt")
LOG_PATH      = os.path.join(CONFIG["output_dir"], "training_log.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'✅ Config OK | Output → {CONFIG["output_dir"]} | Device: {device}')

In [ ]:
# ── CELL 5: Load & Chuẩn bị dữ liệu ─────────────────────────────────────────
print('📂 Loading data...')
df = pd.read_parquet(CONFIG["data_path"])

print(f'  Tổng records : {len(df):,}')
print(f'  Scam=True    : {df["label"].sum():,}  ({df["label"].mean()*100:.3f}%)')
print(f'  Scam=False   : {(~df["label"]).sum():,}')

# --- Feature engineering ---
df['label']             = df['label'].astype(int)
df['has_url']           = df['has_url'].astype(float)
df['verified']          = df['verified'].astype(float)
# Log-normalize n_subscribers
max_subs = df['n_subscribers'].max()
df['n_subs_norm']       = np.log1p(df['n_subscribers']) / np.log1p(max_subs)
df['cleaned_text']      = df['cleaned_text'].fillna('').astype(str)

# --- Train / Val / Test split (stratified) ---
holdout = CONFIG["val_size"] + CONFIG["test_size"]
train_df, temp_df = train_test_split(
    df, test_size=holdout,
    random_state=CONFIG["seed"], stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5,
    random_state=CONFIG["seed"], stratify=temp_df['label']
)

print(f'\n  Train : {len(train_df):,} | scam={train_df["label"].sum()}')
print(f'  Val   : {len(val_df):,}   | scam={val_df["label"].sum()}')
print(f'  Test  : {len(test_df):,}  | scam={test_df["label"].sum()}')

In [ ]:
# ── CELL 6: Dataset & DataLoader ──────────────────────────────────────────────
print('🔤 Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

class ScamDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.texts   = df['cleaned_text'].tolist()
        self.has_url = df['has_url'].tolist()
        self.verified= df['verified'].tolist()
        self.n_subs  = df['n_subs_norm'].tolist()
        self.labels  = df['label'].tolist()
        self.tok     = tokenizer
        self.max_len = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'metadata'      : torch.tensor(
                [self.has_url[idx], self.verified[idx], self.n_subs[idx]],
                dtype=torch.float
            ),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds = ScamDataset(train_df, tokenizer, CONFIG['max_length'])
val_ds   = ScamDataset(val_df,   tokenizer, CONFIG['max_length'])
test_ds  = ScamDataset(test_df,  tokenizer, CONFIG['max_length'])

# WeightedRandomSampler: oversample lớp scam (minority)
labels_arr   = np.array(train_df['label'].tolist())
class_counts = np.bincount(labels_arr)           # [n_neg, n_pos]
class_w      = 1.0 / class_counts                # weight tỉ lệ nghịch
sample_w     = class_w[labels_arr]               # weight cho từng sample
sampler      = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'],
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size']*2,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size']*2,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Datasets ready')
print(f'   Steps per epoch ≈ {len(train_loader):,}')

In [ ]:
# ── CELL 7: Model ─────────────────────────────────────────────────────────────
class ScamClassifier(nn.Module):
    """
    PhoBERT encoder + metadata features → binary classifier
    Input : text token + [has_url, verified, n_subs_norm]
    Output: logit [not_scam, scam]
    """
    def __init__(self, model_name, num_meta=3, num_classes=2):
        super().__init__()
        self.encoder   = AutoModel.from_pretrained(model_name)
        hidden         = self.encoder.config.hidden_size   # 768
        self.head      = nn.Sequential(
            nn.Linear(hidden + num_meta, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, metadata):
        out  = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls  = out.last_hidden_state[:, 0, :]   # [CLS] embedding
        x    = torch.cat([cls, metadata], dim=1)
        return self.head(x)


print('🤖 Building model...')
model = ScamClassifier(CONFIG['model_name'])

# Áp dụng LoRA vào các attention layers
lora_cfg = LoraConfig(
    task_type     = TaskType.FEATURE_EXTRACTION,
    r             = CONFIG['lora_r'],
    lora_alpha    = CONFIG['lora_alpha'],
    lora_dropout  = CONFIG['lora_dropout'],
    target_modules= ['query', 'key', 'value'],
    bias          = 'none'
)
model.encoder = get_peft_model(model.encoder, lora_cfg)
model.encoder.print_trainable_parameters()
model = model.to(device)
print('✅ Model ready')

In [ ]:
# ── CELL 8: Focal Loss + Optimizer + Scheduler ───────────────────────────────
class FocalLoss(nn.Module):
    """
    Focal Loss – giảm đóng góp của samples dễ (majority class)
    gamma  : focusing parameter (2 là chuẩn)
    alpha  : weight cho class Scam=True (đặt cao để ưu tiên recall)
    """
    def __init__(self, gamma=2.0, alpha=0.85):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        ce   = nn.functional.cross_entropy(logits, targets, reduction='none')
        pt   = torch.exp(-ce)
        fl   = (1 - pt) ** self.gamma * ce
        a_t  = torch.where(targets == 1,
                            torch.full_like(ce, self.alpha),
                            torch.full_like(ce, 1 - self.alpha))
        return (a_t * fl).mean()


criterion = FocalLoss(gamma=CONFIG['focal_gamma'], alpha=CONFIG['focal_alpha'])

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG['learning_rate'], weight_decay=0.01
)

total_steps   = len(train_loader) * CONFIG['epochs']
warmup_steps  = int(total_steps * CONFIG['warmup_ratio'])
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps  = warmup_steps,
    num_training_steps= total_steps
)

scaler = GradScaler()   # mixed precision

print(f'✅ Optimizer ready | Total steps: {total_steps:,} | Warmup: {warmup_steps:,}')

In [ ]:
# ── CELL 9: Checkpoint helpers ────────────────────────────────────────────────
def save_checkpoint(epoch, global_step, best_f1, history):
    torch.save({
        'epoch'          : epoch,
        'global_step'    : global_step,
        'model_state'    : model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'scaler_state'   : scaler.state_dict(),
        'best_f1'        : best_f1,
    }, CKPT_PATH)
    with open(LOG_PATH, 'w') as f:
        json.dump(history, f, indent=2, ensure_ascii=False)
    print(f'  💾 Checkpoint saved (epoch={epoch+1}, step={global_step:,}, best_f1={best_f1:.4f})')


def load_checkpoint():
    """Returns (start_epoch, start_step, best_f1, history)"""
    if not os.path.exists(CKPT_PATH):
        print('⚡ Không tìm thấy checkpoint → bắt đầu mới.')
        return 0, 0, 0.0, []

    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    scaler.load_state_dict(ckpt['scaler_state'])

    history = json.load(open(LOG_PATH)) if os.path.exists(LOG_PATH) else []
    epoch  = ckpt['epoch']
    step   = ckpt['global_step']
    best_f1= ckpt['best_f1']
    print(f'✅ Resume từ epoch={epoch+1}, step={step:,}, best_f1={best_f1:.4f}')
    return epoch, step, best_f1, history


print('✅ Checkpoint helpers ready')

In [ ]:
# ── CELL 10: Evaluate ─────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(loader):
    model.eval()
    total_loss = 0
    preds_all, labels_all = [], []

    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        meta = batch['metadata'].to(device)
        lbl  = batch['label'].to(device)

        with autocast():
            logits = model(ids, mask, meta)
            loss   = criterion(logits, lbl)

        total_loss += loss.item()
        preds_all.extend(torch.argmax(logits, 1).cpu().tolist())
        labels_all.extend(lbl.cpu().tolist())

    model.train()
    return {
        'loss'     : total_loss / len(loader),
        'f1'       : f1_score(labels_all, preds_all, average='binary', zero_division=0),
        'precision': precision_score(labels_all, preds_all, average='binary', zero_division=0),
        'recall'   : recall_score(labels_all, preds_all, average='binary', zero_division=0),
        'accuracy' : sum(p==l for p,l in zip(preds_all,labels_all)) / len(labels_all),
    }

print('✅ Evaluate helper ready')

In [ ]:
# ── CELL 11: TRAINING LOOP ────────────────────────────────────────────────────
start_epoch, global_step, best_f1, history = load_checkpoint()
patience_counter = 0
steps_per_epoch  = len(train_loader)

print(f'\n{'='*65}')
print(f' Training | Epochs {start_epoch+1} → {CONFIG["epochs"]} | Steps/epoch ≈ {steps_per_epoch:,}')
print(f' Save every {CONFIG["save_every_steps"]:,} steps | Eval every {CONFIG["eval_every_steps"]:,} steps')
print(f'{'='*65}\n')

model.train()
epoch_loss = 0

for epoch in range(start_epoch, CONFIG['epochs']):
    epoch_start = time.time()

    for step_in_epoch, batch in enumerate(train_loader):

        # Nếu resume giữa chừng: skip các step đã làm
        steps_done_in_epoch = global_step - epoch * steps_per_epoch
        if step_in_epoch < steps_done_in_epoch:
            continue

        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        meta = batch['metadata'].to(device)
        lbl  = batch['label'].to(device)

        optimizer.zero_grad()
        with autocast():
            logits = model(ids, mask, meta)
            loss   = criterion(logits, lbl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss  += loss.item()
        global_step += 1

        # --- Log mỗi 500 steps ---
        if global_step % 500 == 0:
            avg_loss = epoch_loss / (step_in_epoch + 1)
            lr_now   = scheduler.get_last_lr()[0]
            elapsed  = time.time() - epoch_start
            print(f'  Epoch {epoch+1} | Step {global_step:,} | loss={avg_loss:.4f} | lr={lr_now:.2e} | {elapsed:.0f}s')

        # --- Validate + checkpoint mỗi eval_every_steps ---
        if global_step % CONFIG['eval_every_steps'] == 0:
            m   = evaluate(val_loader)
            ts  = datetime.now().isoformat()
            row = {
                'epoch'      : epoch + 1,
                'global_step': global_step,
                'train_loss' : epoch_loss / (step_in_epoch + 1),
                **{f'val_{k}': v for k,v in m.items()},
                'timestamp'  : ts
            }
            history.append(row)
            print(f'\n  📊 EVAL  step={global_step:,}')
            print(f'      Val Loss={m["loss"]:.4f} | F1={m["f1"]:.4f} | Prec={m["precision"]:.4f} | Recall={m["recall"]:.4f}')

            if m['f1'] > best_f1:
                best_f1 = m['f1']
                torch.save(model.state_dict(), BEST_MODEL_PATH)
                print(f'      🏆 New best F1={best_f1:.4f} → saved best_model.pt')
                patience_counter = 0
            else:
                patience_counter += 1
                print(f'      ⏳ No improvement ({patience_counter}/{CONFIG["early_stop_patience"]})')
                if patience_counter >= CONFIG['early_stop_patience']:
                    print('\n🛑 Early stopping triggered!')
                    save_checkpoint(epoch, global_step, best_f1, history)
                    raise SystemExit  # thoát vòng lặp sạch

        # --- Checkpoint định kỳ ---
        if global_step % CONFIG['save_every_steps'] == 0:
            save_checkpoint(epoch, global_step, best_f1, history)

    # --- Cuối epoch ---
    print(f'\n✔ Epoch {epoch+1} done | avg_loss={epoch_loss/steps_per_epoch:.4f} | {time.time()-epoch_start:.0f}s')
    epoch_loss = 0
    save_checkpoint(epoch, global_step, best_f1, history)

print(f'\n{'='*65}')
print(f' Training complete! Best Val F1 = {best_f1:.4f}')
print(f'{'='*65}')

In [ ]:
# ── CELL 12: Đánh giá trên Test Set ──────────────────────────────────────────
print('📥 Loading best model...')
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

test_m = evaluate(test_loader)

print('\n' + '='*40)
print(' TEST SET RESULTS')
print('='*40)
for k, v in test_m.items():
    print(f'  {k:12}: {v:.4f}')
print('='*40)

In [ ]:
# ── CELL 13: Plot training history ────────────────────────────────────────────
import matplotlib.pyplot as plt

if history:
    steps  = [h['global_step'] for h in history]
    f1s    = [h['val_f1']      for h in history]
    losses = [h['val_loss']    for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(steps, f1s,    marker='o')
    axes[0].set_title('Val F1 Score')
    axes[0].set_xlabel('Global Step')
    axes[0].set_ylabel('F1')
    axes[0].grid(True)

    axes[1].plot(steps, losses, marker='o', color='orange')
    axes[1].set_title('Val Loss')
    axes[1].set_xlabel('Global Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True)

    plt.tight_layout()
    plot_path = os.path.join(CONFIG['output_dir'], 'training_curve.png')
    plt.savefig(plot_path)
    plt.show()
    print(f'✅ Plot saved → {plot_path}')
else:
    print('No history to plot.')

In [ ]:
# ── CELL 14: Inference (dùng sau khi train) ───────────────────────────────────
def predict(texts: list[str], has_urls=None, verified=None, n_subs=None):
    """Trả về list tuple (label, confidence)"""
    if has_urls is None:
        has_urls = [0.0] * len(texts)
    if verified is None:
        verified = [0.0] * len(texts)
    if n_subs is None:
        n_subs   = [0.0] * len(texts)

    model.eval()
    results = []

    for i in range(0, len(texts), 32):
        batch_texts = texts[i:i+32]
        enc = tokenizer(
            batch_texts, max_length=CONFIG['max_length'],
            padding='max_length', truncation=True, return_tensors='pt'
        )
        meta = torch.tensor(
            [[has_urls[j], verified[j], n_subs[j]] for j in range(i, min(i+32, len(texts)))],
            dtype=torch.float
        ).to(device)

        with torch.no_grad(), autocast():
            logits = model(enc['input_ids'].to(device), enc['attention_mask'].to(device), meta)
            probs  = torch.softmax(logits, dim=1)

        for prob in probs:
            label = 'SCAM' if prob[1] > 0.5 else 'NOT SCAM'
            conf  = prob[1].item()
            results.append((label, conf))

    return results

# Test thử
samples = [
    "Chúc mừng bạn đã trúng thưởng 100 triệu đồng! Liên hệ ngay để nhận giải.",
    "Hôm nay trời đẹp quá, đi chơi thôi mọi người ơi!",
    "Đầu tư tiền ảo sinh lời 200%/tháng, cam kết hoàn vốn 100%."
]
for text, (label, conf) in zip(samples, predict(samples)):
    print(f'[{label} | {conf:.3f}] {text[:60]}...')